# Assignment Customer Satisfaction and Sentiment Analysis


## Objective

You are a data analyst of a consulting company that provides customer insight regarding multiple ticketing system, such as JIRA and Zoho Desk. Your team gather surveys to customers regarding their ticketing system's performance. Your role in the team is to gather reports regarding customer satisfaction and sentiment analysis into a single dashboard and present your insight.

Analyze the following metrics and other insight you can find in the dataset:

- Survey response rate
- Customer Satisfaction score (CSAT)
- Customer Effort Score (CES)
- Net Promoter Score (NPS)
- Sentiment Analysis



## Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.options.display.float_format = "{:.2f}".format

# set styk=ling for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(style='whitegrid')

### Access to Drive

Write where you put the data in google drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# define the location
path_data = '/content/drive/MyDrive/Dibimbing_DS/assignment/customer_satisfaction/dataset/'

Mounted at /content/drive/


### Read Data

Read the file **assignment_ticket_system_review.csv**

In [ ]:
# Read Data
df_ticket_system = pd.read_csv(path_data + 'assignment_ticket_system_review.csv'  )

The following is the dictionary for the data, survey is only valid if all of the survey questions and text review is not blank (null):

**General Information**
- id_survey: identifier for each survey
- date_of_survey: date of survey taken
- ticket_system: The name of the ticket system being reviewed (e.g. Zoho Desk)

**Survey Questions**
- overall_rating: The overall satisfaction rating given by the reviewer, ranging from 1 to 5
- customer_service: The satisfaction rating for the customer service provided by the ticket system, ranging from 1 to 5.
- features: The satisfaction rating for the features of the ticket system, ranging from 1 to 5
- value_for_money: The satisfaction rating for the value for money provided by the ticket system, ranging from 1 to 5
- ease_of_use: The rating for how easy the ticket system is to use, ranging from 1 to 5
- likelihood_to_recommend: The likelihood that the reviewer would recommend the ticket system to others, ranging from 1 to 10
- overall_text: The full text of the overall review, providing detailed feedback on the ticket system.


In [ ]:
# Check the type of data
df_ticket_system.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id_survey                1462 non-null   object 
 1   date_of_survey           1462 non-null   object 
 2   ticket_system            1462 non-null   object 
 3   overall_rating           787 non-null    float64
 4   customer_service         787 non-null    float64
 5   features                 787 non-null    float64
 6   value_for_money          787 non-null    float64
 7   ease_of_use              787 non-null    float64
 8   likelihood_to_recommend  787 non-null    float64
 9   overall_text             787 non-null    object 
dtypes: float64(6), object(4)
memory usage: 114.3+ KB


In [ ]:
# check 10 rows in random
df_ticket_system.sample(10)

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text
895,T_02846,2024-12-03,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
617,T_01258,2024-11-02,Jira Service Management,NaN,NaN,NaN,NaN,NaN,NaN,NaN
742,T_00730,2024-10-19,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
418,T_01885,2024-11-15,Freshdesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
927,T_00569,2024-10-15,Freshdesk,5.00,4.00,4.00,5.00,5.00,7.00,Centralization of the support. A place to foll...
16,T_02569,2024-11-29,Freshdesk,5.00,3.00,5.00,5.00,5.00,10.00,I have had a fantastic experience with both th...
1402,T_00179,2024-10-05,Freshdesk,4.00,3.00,4.00,4.00,4.00,5.00,Customer support has been great. But I definit...
515,T_02294,2024-11-23,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN
689,T_01209,2024-11-01,Jira Service Management,NaN,NaN,NaN,NaN,NaN,NaN,NaN
465,T_04347,2024-12-23,Zendesk,5.00,2.00,5.00,4.00,3.00,8.00,ZenDesk is a solid cloud-based tool and very g...


### Data Cleansing

Convert the date column into proper date_time format.

In [ ]:
# Convert data type
df_ticket_system['date_of_survey'] =  pd.to_datetime(df_ticket_system['date_of_survey'])

#### Data Missing

**We acknowledge that the data has a missing value, but we don't apply any treatment (fill with something or remove the missing value) because it will be used for calculating a response rate.**

## Survey Analysis

In [ ]:
# statistic descriptive
df_ticket_system.describe()

,date_of_survey,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend
count,1462,787.00,787.00,787.00,787.00,787.00,787.00
mean,2024-11-22 10:37:15.841313024,4.56,3.37,4.42,4.38,4.47,7.61
min,2024-10-01 00:00:00,1.00,-1.00,1.00,1.00,1.00,-1.00
25%,2024-11-02 06:00:00,4.00,3.00,4.00,4.00,4.00,7.00
50%,2024-11-28 00:00:00,5.00,3.00,5.00,5.00,5.00,8.00
75%,2024-12-14 00:00:00,5.00,4.00,5.00,5.00,5.00,9.00
max,2024-12-30 00:00:00,5.00,5.00,5.00,5.00,5.00,10.00
std,NaN,0.64,1.14,0.72,0.82,0.73,1.72


based on the data information, the score or rating in range 1 to 5. So, these rows will be manipulate by multiplying with -1

In [ ]:
df_ticket_system[df_ticket_system['customer_service'] < 0]

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text
183,T_04025,2024-12-19,Zoho Desk,3.00,-1.00,3.00,3.00,5.00,5.00,There is def. a lot we could use. We are still...
813,T_02534,2024-11-28,Zendesk,4.00,-1.00,4.00,3.00,5.00,1.00,Pros:It has a lot of features and has been aro...


In [ ]:
df_ticket_system.loc[df_ticket_system['customer_service'] < 0, 'customer_service'] = df_ticket_system['customer_service'] * -1

In [ ]:
# rechecking
df_ticket_system[df_ticket_system['customer_service'] < 0]

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,fill_rating


In [ ]:
df_ticket_system[df_ticket_system['likelihood_to_recommend'] < 0]

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,fill_rating
938,T_04185,2024-12-21,Freshdesk,2.00,0.00,4.00,2.00,4.00,-1.00,Pros:The platform is relatively user friendly ...,Rated


In [ ]:
# multiplying by -1 for row that contains value below than 0
df_ticket_system.loc[df_ticket_system['likelihood_to_recommend'] < 0, 'likelihood_to_recommend'] = df_ticket_system['likelihood_to_recommend'] * -1

In [ ]:
# checkpoint
df_ticket_system[df_ticket_system['likelihood_to_recommend'] < 0]

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,fill_rating


### Response Rate

Start by analyzing how many customers has filled the survey, indicated by whether the overall_rating is not blank.

In [ ]:
# How many customer responded to the survey?
df_ticket_system['fill_rating'] = np.where(df_ticket_system['overall_rating'].isnull(), 'Not Rated', 'Rated' )

df_ticket_system.value_counts('fill_rating', normalize=True).reset_index()

,fill_rating,proportion
0,Rated,0.54
1,Not Rated,0.46


In [ ]:
# check null values
print(f"Not Rated = {df_ticket_system['overall_rating'].isnull().sum()/len(df_ticket_system):.2f}")
print(f"Rated = {1-(df_ticket_system['overall_rating'].isnull().sum() /len(df_ticket_system)):.2f}")

Not Rated = 0.46
Rated = 0.54


* Based on percentages, users or consumers who fill out reviews or ratings are more likely than those who do not.
* However, the gap is not too big or the portion between the two is quite balanced.

Create a new dataframe that consists only of those who have responded the survey to simplify calculating the CSAT, CES, and NPS Score.

In [ ]:
# Responded Customer
responded_customer = df_ticket_system[df_ticket_system['fill_rating'] == 'Rated'].copy()

### CSAT Score

Measure the customer's overall satisfaction score (CSAT) with the following formula:

$$
CSAT = \frac{\Sigma\ total\ satisfaction\ score}{number\ of\ responded\ customer \times \max\ rating}
$$

The max rating is inserted to convert the CSAT score into percentage.

CSAT score can be classified into categories based on the result. There is no absolute threshold for each categories but the following is the common threshold:

- \>= 90%: Excellent
- 75%-90%: Good
- 60-75%: Fair
- \<60%: Poor

In [ ]:
# CSAT Score
max_rating = df_ticket_system['overall_rating'].max()
n_data = (responded_customer.shape[0] * max_rating)

csat_score = responded_customer['overall_rating'].sum() / n_data

# print the result
print(f"Overall CSAT Score: {(csat_score * 100):.2f}%")

Overall CSAT Score: 91.18%


* The vast majority of customers had a very positive experience and were very satisfied with the product or service provided.
* This means that of all the customers who filled out almost 91% were satisfied with the ticket system.

We can calculate the other indicators, like customer service, features, and value for money

Measure the satisfaction score for the following attributes:

- customer service
- features
- value for money

In [ ]:
# Satisfaction Score for Attributes
score_customer_service = responded_customer['customer_service'].sum() / n_data
score_features = responded_customer['features'].sum() / n_data
score_value_for_money = responded_customer['value_for_money'].sum() / n_data

print(f'Overall CSAT Score: { (csat_score * 100):.1f}%')
print(f'The Customer Service CSAT score: { (score_customer_service * 100):.1f}%')
print(f'The Features CSAT score: { (score_features * 100):.1f}%')
print(f'The Value for Moeny CSAT score: { (score_value_for_money * 100):.1f}%')

Overall CSAT Score: 91.2%
The Customer Service CSAT score: 67.4%
The Features CSAT score: 88.3%
The Value for Moeny CSAT score: 87.6%


* Among other features, customer service has the lowest score.
* This means that the level of customer satisfaction regarding customer service needs to be improved because the services provided have not met user expectations.
* As for Features, there seems to be no problem. Based on the score value, users are generally satisfied with the features provided, which meet their needs.
* Likewise, in terms of value for money, the score indicates that users perceive the money they spend as being in line with what they receive or what is offered.

### CES Score

Measure CES with the following formula


$$
CES = \frac{\Sigma\ total\ effort\ score}{number\ of\ responded\ customer \times \max\ rating}
$$

In [ ]:
# CES Score
# We ease of use column
max_rating = df_ticket_system['ease_of_use'].max()
n_data = (responded_customer.shape[0] * max_rating)

CES_ease_of_use = responded_customer['ease_of_use'].sum() / n_data

# print the result
print(f'Ease of use CES Score: {(CES_ease_of_use*100):.2f}%')

Ease of use CES Score: 89.48%


* The majority of customers find the products/services provided easy to use.
* In other words, the product design, systems, processes, and features do not require too much effort to use or understand.

### NPS Score

To calculate the NPS score, first we must convert the **would_you_recommend** column into proper NPS Category based on the rating value:

* Promoter: Rating 9-10
* Passive: Rating 7-8
* Detractor: Rating < 7

In [ ]:
responded_customer.head()

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,fill_rating
1,T_00229,2024-10-06,Zendesk,3.00,4.00,3.00,3.00,2.00,6.00,Customer tickets managements,Rated
2,T_04527,2024-12-26,Zoho Desk,5.00,5.00,5.00,5.00,5.00,8.00,"After 6 months of using the Zoho desk, we shif...",Rated
4,T_00644,2024-10-17,Zendesk,5.00,3.00,4.00,5.00,5.00,6.00,Pros:Zendesk has always been one of the go-to ...,Rated
6,T_04682,2024-12-28,Zoho Desk,5.00,4.00,5.00,5.00,5.00,8.00,It has been very useful so far to integrate mu...,Rated
8,T_01238,2024-11-02,Freshdesk,4.00,4.00,4.00,5.00,4.00,8.00,Pros:It's easy to use and very intuitive.We ha...,Rated


In [ ]:
responded_customer.info()

<class 'pandas.core.frame.DataFrame'>
Index: 787 entries, 1 to 1461
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_survey                787 non-null    object        
 1   date_of_survey           787 non-null    datetime64[ns]
 2   ticket_system            787 non-null    object        
 3   overall_rating           787 non-null    float64       
 4   customer_service         787 non-null    float64       
 5   features                 787 non-null    float64       
 6   value_for_money          787 non-null    float64       
 7   ease_of_use              787 non-null    float64       
 8   likelihood_to_recommend  787 non-null    float64       
 9   overall_text             787 non-null    object        
 10  fill_rating              787 non-null    object        
dtypes: datetime64[ns](1), float64(6), object(4)
memory usage: 73.8+ KB


In [ ]:
responded_customer['likelihood_to_recommend'].value_counts()

,count
likelihood_to_recommend,
8.00,224
9.00,174
7.00,155
6.00,82
10.00,77
5.00,34
4.00,18
3.00,10
1.00,6


In [ ]:
# Category NPS
nps_value = ['Promoter', 'Passive', 'Detractor']
nps_condition = [
    responded_customer['likelihood_to_recommend'] >= 9,
    (responded_customer['likelihood_to_recommend'] >= 7) & (responded_customer['likelihood_to_recommend'] < 9),
    responded_customer['likelihood_to_recommend'] < 7
]

responded_customer['nps_category'] = pd.cut(
    responded_customer['likelihood_to_recommend'],
    bins=[-np.inf, 6.5, 8.5, np.inf], #lower limit = -np.inf - 6, middle limit = 7-8, upper limit = 9-10
    labels=['Detractor', 'Passive', 'Promoter'],
    include_lowest=True
)

responded_customer.value_counts('nps_category', normalize = True)

,proportion
nps_category,
Passive,0.48
Promoter,0.32
Detractor,0.20



Calculate the NPS Score with the following formula

$$
NPS = \frac{Promoter - Detractor}{Total\ Survey\ Responded}
$$

In [ ]:
# NPS Score
NPS_agg= responded_customer.value_counts('nps_category').reset_index()

NPS_agg_promoter = NPS_agg[NPS_agg['nps_category']=='Promoter']['count'].item()
NPS_agg_detractor = NPS_agg[NPS_agg['nps_category']=='Detractor']['count'].item()
NPS_score = (NPS_agg_promoter-NPS_agg_detractor)/len(responded_customer)

print(f'NPS Score: {(NPS_score*100):.2f}')

NPS Score: 11.94


NPS score can be ranging from -100 (when all customers are detractor) to 100 (when all customers are promoter).

NPS Score can be classified into categories based on the following threshold:

- \>= 70: Excellent
- 50-69: Very Good
- 30-49: Good
- 0-29: Average
- \< 0: Poor

* In general, NPS is of a moderate size as seen from the user base of “Promoter” more than “Detractor”, which means that users who are loyal and want to recommend more than users who are not happy/satisfied.
* However, the majority dominance of the “Passive” category indicates that there is potential for customers to be satisfied, but will not recommend (neutral)

## Sentiment Analysis

Create a new dataframe with no blank overall_text.

In [ ]:
df_ticket_system.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_survey                1462 non-null   object        
 1   date_of_survey           1462 non-null   datetime64[ns]
 2   ticket_system            1462 non-null   object        
 3   overall_rating           787 non-null    float64       
 4   customer_service         787 non-null    float64       
 5   features                 787 non-null    float64       
 6   value_for_money          787 non-null    float64       
 7   ease_of_use              787 non-null    float64       
 8   likelihood_to_recommend  787 non-null    float64       
 9   overall_text             787 non-null    object        
 10  fill_rating              1462 non-null   object        
dtypes: datetime64[ns](1), float64(6), object(4)
memory usage: 125.8+ KB


In [ ]:
df_ticket_system.sample(10)

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,fill_rating
1224,T_01124,2024-10-29,Zendesk,4.00,4.00,4.00,4.00,4.00,6.00,It's good overall. They just need to work on g...,Rated
1181,T_00725,2024-10-19,Freshdesk,5.00,5.00,5.00,4.00,4.00,7.00,Ottimo Software per helpdesk .tutti i ticket m...,Rated
666,T_04132,2024-12-20,Zendesk,4.00,3.00,4.00,3.00,5.00,5.00,"Pros:Zendesk is a quality, best of class solut...",Rated
36,T_00457,2024-10-12,Zoho Desk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated
1300,T_00618,2024-10-17,Zoho Desk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated
739,T_02289,2024-11-23,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated
977,T_00764,2024-10-20,Zendesk,5.00,3.00,5.00,4.00,5.00,9.00,I was able to set up Macros and were able to u...,Rated
104,T_02627,2024-11-30,Freshdesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated
55,T_04346,2024-12-23,Freshdesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated
819,T_04101,2024-12-20,Zoho Desk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated


In [ ]:
# Create new dataframe
df_review_ticket_system_clean = df_ticket_system.dropna(axis=0, subset='overall_text')[['id_survey', 'overall_text']].copy()

df_review_ticket_system_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 787 entries, 1 to 1461
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_survey     787 non-null    object
 1   overall_text  787 non-null    object
dtypes: object(2)
memory usage: 18.4+ KB


In [ ]:
df_review_ticket_system_clean.head(5)

,id_survey,overall_text
1,T_00229,Customer tickets managements
2,T_04527,"After 6 months of using the Zoho desk, we shif..."
4,T_00644,Pros:Zendesk has always been one of the go-to ...
6,T_04682,It has been very useful so far to integrate mu...
8,T_01238,Pros:It's easy to use and very intuitive.We ha...


### Text Cleansing

In order to get more accurate sentiment, several text cleansing need to be done. However, in most of recent sentiment analysis models and algorithm, the only text cleansing needed are as follows:

* Clean double whitespace
* Clean URL/website
* Clean username (mostly in social media or digital text)

In [ ]:
import re

def cleansing_text(x):
  # clean double whitespace
  out_text = ' '.join(x.split())

  # clean url
  out_text = re.sub(r"http\S+|www\S+|https\S+", 'http', out_text)

  # clean username
  out_text = re.sub(r"@\S+", '@user', out_text)

  return(out_text)

cleansing_text(" Doesn't  dissapoint. The car       was great. It was the best car rental experiences I've had! Salute to @jone who recommend https:/rental.com")

"Doesn't dissapoint. The car was great. It was the best car rental experiences I've had! Salute to @user who recommend http"

In [ ]:
# apply cleansing to review
df_review_ticket_system_clean['text_clean'] = df_review_ticket_system_clean['overall_text'].apply(cleansing_text)

df_review_ticket_system_clean.head(10)

,id_survey,overall_text,text_clean
1,T_00229,Customer tickets managements,Customer tickets managements
2,T_04527,"After 6 months of using the Zoho desk, we shif...","After 6 months of using the Zoho desk, we shif..."
4,T_00644,Pros:Zendesk has always been one of the go-to ...,Pros:Zendesk has always been one of the go-to ...
6,T_04682,It has been very useful so far to integrate mu...,It has been very useful so far to integrate mu...
8,T_01238,Pros:It's easy to use and very intuitive.We ha...,Pros:It's easy to use and very intuitive.We ha...
9,T_00355,Pros:We have connected Zendesk with a lot of o...,Pros:We have connected Zendesk with a lot of o...
12,T_02440,"Its best, but need extra improvement from UI Side","Its best, but need extra improvement from UI Side"
15,T_03213,"Served us incredibly well for years, the macro...","Served us incredibly well for years, the macro..."
16,T_02569,I have had a fantastic experience with both th...,I have had a fantastic experience with both th...
19,T_03530,Freshdesk has enabled us to manage customer ti...,Freshdesk has enabled us to manage customer ti...


### Sentiment Analysis

Create a sentiment categories using algorithm of your own choice.

I use this model to transform, predict, and generate sentiment from the review.

https://huggingface.co/tabularisai/multilingual-sentiment-analysis

In [ ]:
# Sentiment Algorithm
%%capture
!pip install transformers
from transformers import pipeline

# Load the classification pipeline with the specified model
sentiment_pipeline = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

Check the number of data by sentiment.

In [ ]:
%%time
# Prediksi sentimen
transformer_results = sentiment_pipeline(df_review_ticket_system_clean['text_clean'].tolist())

# store sentiment into df
df_review_ticket_system_clean['sentiment'] = [res['label'] for res in transformer_results]

# store score into df
df_review_ticket_system_clean['sentiment_probability'] = [res['score'] for res in transformer_results]

CPU times: user 1min 41s, sys: 235 ms, total: 1min 41s
Wall time: 2min 17s


In [ ]:
df_review_ticket_system_clean.head(10)

,id_survey,overall_text,text_clean,sentiment,sentiment_probability
1,T_00229,Customer tickets managements,Customer tickets managements,Neutral,0.45
2,T_04527,"After 6 months of using the Zoho desk, we shif...","After 6 months of using the Zoho desk, we shif...",Neutral,0.38
4,T_00644,Pros:Zendesk has always been one of the go-to ...,Pros:Zendesk has always been one of the go-to ...,Very Positive,0.41
6,T_04682,It has been very useful so far to integrate mu...,It has been very useful so far to integrate mu...,Neutral,0.39
8,T_01238,Pros:It's easy to use and very intuitive.We ha...,Pros:It's easy to use and very intuitive.We ha...,Very Positive,0.34
9,T_00355,Pros:We have connected Zendesk with a lot of o...,Pros:We have connected Zendesk with a lot of o...,Positive,0.40
12,T_02440,"Its best, but need extra improvement from UI Side","Its best, but need extra improvement from UI Side",Neutral,0.49
15,T_03213,"Served us incredibly well for years, the macro...","Served us incredibly well for years, the macro...",Very Positive,0.52
16,T_02569,I have had a fantastic experience with both th...,I have had a fantastic experience with both th...,Very Positive,0.45
19,T_03530,Freshdesk has enabled us to manage customer ti...,Freshdesk has enabled us to manage customer ti...,Positive,0.53


In [ ]:
# check the logic
print(df_review_ticket_system_clean.loc[1, 'text_clean']) # this result is neutral

Customer tickets managements


In [ ]:
# check the logic
print(df_review_ticket_system_clean.loc[2, 'text_clean']) # this result is neutral

After 6 months of using the Zoho desk, we shifted to different software, but my experience using Zoho was great! Indeed, I would still recommend using Zoho after all.


In [ ]:
# check the logic
print(df_review_ticket_system_clean.loc[4, 'text_clean']) #this result is very positive

Pros:Zendesk has always been one of the go-to solutions for helpdesk software, but they've really streamlined their system over the last few years. It's simple and easy to use with straightforward options for even those who are new to using a helpdesk system.Zendesk is basically an email support system, where all emails sent will be routed to your Zendesk dashboard and show up as support tickets. Tickets are tagged with its own ID and even cross-referenced to see if a particular sender has submitted any tickets prior, so it's easy to check through the support history. There are also pre-defined responses, allowing for quick and easy replies for typical queries.Of course, these are actually what you might believe every helpdesk system should provide, but you'll be surprised at how much you have to pay and that some don't even have these options at all. Zendesk is actually very affordable and is easily scalable.


Based on the result and some a text to see full, we'll use this for analyzing in the further. Couple note to take look:
1. Model appears to be good with neutral review as in review in index 1 "Customer ticket management".
2. Similar with very positive result on review in index 4, 15, 16, etc.
3. The probability or sentiment score is the result of the highest probability of the class.
4. The score is not division of the number of categories, for example in this model we have 5 categories, it doesn't mean dengan bins of categories 0.20.
5. So the interpretation of probability is not like this, if probability < 0.20 then we'll have sentiment very negative".
6. The probability is only for the text, meaning the score of sentiment from review is highest among the other classes.
7. Each probability is uniquely calculated based on the text input and the model's 'knowledge.' For example, a text might have a probability of 0.80 for Neutral, 0.10 for Positive, and very small probabilities for other classes.

In [ ]:
# check the total among sentiment
df_review_ticket_system_clean['sentiment'].value_counts()

,count
sentiment,
Positive,335
Very Positive,213
Neutral,211
Negative,16
Very Negative,12


* The values of CSAT, NPS, and other attributes reflect the number of sentiment categories obtained from user-given reviews
* The majority of reviews contain positive and very positive sentiments.
* However, reviews with neutral nuances were also found to be not small

## Finalize Data for Reporting

Save the review data with NPS category and sentiment information to new csv for the dashboard.

In [ ]:
# Save Data
df_review_ticket_system_final = df_ticket_system.merge(
    df_review_ticket_system_clean[['id_survey', 'sentiment']],
    on = 'id_survey',
    how = 'left'
)

df_review_ticket_system_final.head(10)

,id_survey,date_of_survey,ticket_system,overall_rating,customer_service,features,value_for_money,ease_of_use,likelihood_to_recommend,overall_text,fill_rating,sentiment
0,T_02161,2024-11-20,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated,NaN
1,T_00229,2024-10-06,Zendesk,3.00,4.00,3.00,3.00,2.00,6.00,Customer tickets managements,Rated,Neutral
2,T_04527,2024-12-26,Zoho Desk,5.00,5.00,5.00,5.00,5.00,8.00,"After 6 months of using the Zoho desk, we shif...",Rated,Neutral
3,T_03190,2024-12-08,Zoho Desk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated,NaN
4,T_00644,2024-10-17,Zendesk,5.00,3.00,4.00,5.00,5.00,6.00,Pros:Zendesk has always been one of the go-to ...,Rated,Very Positive
5,T_02868,2024-12-03,Zendesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated,NaN
6,T_04682,2024-12-28,Zoho Desk,5.00,4.00,5.00,5.00,5.00,8.00,It has been very useful so far to integrate mu...,Rated,Neutral
7,T_03968,2024-12-18,Freshdesk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not Rated,NaN
8,T_01238,2024-11-02,Freshdesk,4.00,4.00,4.00,5.00,4.00,8.00,Pros:It's easy to use and very intuitive.We ha...,Rated,Very Positive
9,T_00355,2024-10-09,Zendesk,5.00,4.00,4.00,4.00,4.00,7.00,Pros:We have connected Zendesk with a lot of o...,Rated,Positive


In [ ]:
df_review_ticket_system_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_survey                1462 non-null   object        
 1   date_of_survey           1462 non-null   datetime64[ns]
 2   ticket_system            1462 non-null   object        
 3   overall_rating           787 non-null    float64       
 4   customer_service         787 non-null    float64       
 5   features                 787 non-null    float64       
 6   value_for_money          787 non-null    float64       
 7   ease_of_use              787 non-null    float64       
 8   likelihood_to_recommend  787 non-null    float64       
 9   overall_text             787 non-null    object        
 10  fill_rating              1462 non-null   object        
 11  sentiment                787 non-null    object        
dtypes: datetime64[ns](1), float64(6), 

In [ ]:
# check the column fill_rating to make the output only two classes
df_review_ticket_system_final['fill_rating'].value_counts()

,count
fill_rating,
Rated,787
Not Rated,675


In [ ]:
# # save as csv, and store into path data
# df_review_ticket_system_final.to_csv(path_data + 'ticket_system_processed.csv', index=False)

# save as excel, and store into path data
df_review_ticket_system_final.to_excel(path_data + 'ticket_system_processed.xlsx', index=False)